# Le Gros Chaton — Trajectory SFT (Phase 2b)

Bakes the teacher's agentic behavior (tool calls, self-awareness, creativity) into the 9B weights.

**What this does:**
1. Pulls verified Kimi K3 teacher traces from `mateo0093/le-gros-chaton-traces`
2. Loads Qwen3.5-9B + the 91% Fable5 SFT adapter (`mateo0093/le-gros-chaton-qwen`)
3. Trains on ASSISTANT tokens only (the model learns to *act*, not copy tool output)
4. Uploads the trajectory-SFT adapter to `mateo0093/le-gros-chaton-qwen` as `traj_sft/`

**How to run:**
1. Runtime → Change runtime type → **T4 GPU** (free)
2. Click the key icon in the left sidebar → **Add new secret** → name: `HF_TOKEN`, value: your HF token (needs write access to `mateo0093/le-gros-chaton-qwen`)
3. Run all cells.

**One-command equivalent:** this notebook wraps the standalone script `colab/trajectory_sft.py` — `!python trajectory_sft.py` runs the exact same steps (same model, same assistant-only masking, same upload).

**Fully self-contained** — no repo clone needed.

**GPU options (one line each):**
- **Local RTX 2070 (8GB):** `.venv/bin/python colab/trajectory_sft.py` — needs `.venv` + ~10GB free disk; 4-bit + fp16 + grad-accum 8 fits in 8GB (add `--no-upload` for a local dry run).
- **Colab T4 (free):** `colab run --gpu T4 trajectory_sft.py` (or open this notebook and run all cells).
- **Kaggle:** enable the GPU accelerator, then `HF_TOKEN=hf_xxx python colab/trajectory_sft.py`.
- **Modal:** `modal run colab/trajectory_sft.py` on an A10G image (pip install the HF stack from the notebook's cell 1 first).


In [ ]:
# 0. Install deps (T4, ~2 min). NOTE: keep Colab's preinstalled torch (2.11) —
# forcing torch==2.10 broke its torchvision (torchvision::nms error). Only the
# HF stack + peft need installing.
!pip install -q transformers==5.14.1 tokenizers==0.22.1 peft bitsandbytes datasets accelerate safetensors tiktoken trl
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"
print('deps OK')

In [ ]:
# 1. Config
from google.colab import userdata
import os

HF_TOKEN = userdata.get("HF_TOKEN")  # set in Colab secrets (key icon)
MODEL_NAME = "Qwen/Qwen3.5-9B"
ADAPTER = "mateo0093/le-gros-chaton-qwen"  # 91% Fable5 SFT adapter
TRACES_REPO = "mateo0093/le-gros-chaton-traces"  # verified teacher traces
OUT_REPO = "mateo0093/le-gros-chaton-qwen"  # where traj-SFT adapter goes
TRAJECTORY_CTX = 16384  # long context for tool-use traces
BATCH = 1  # fits T4 with grad-accum 8 = eff batch 8
EPOCHS = 3  # small dataset -> multiple passes
LR = 2e-4

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print('config OK')
print('GPU:', os.popen('nvidia-smi --query-gpu=name --format=csv,noheader').read().strip())

In [ ]:
# 2. Pull the verified teacher traces
from huggingface_hub import hf_hub_download
import json

local = hf_hub_download(repo_id=TRACES_REPO, repo_type="dataset",
                       filename="agent_traces_full.jsonl", token=HF_TOKEN)
traces = [json.loads(l) for l in open(local) if l.strip()]
print(f"Loaded {len(traces)} traces")
print(f"Verified: {sum(1 for t in traces if t.get('verified'))}")
t0 = traces[0]
print(f"Sample: {t0['instance_id']} | turns={t0['turns']} | msgs={len(t0['messages'])}")

In [ ]:
# 3. Load model + 91% SFT adapter in 4-bit (fits T4)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

quant = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4",
)
tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=quant, device_map="auto",
    trust_remote_code=True, torch_dtype=torch.float16,
)
model = PeftModel.from_pretrained(model, ADAPTER)
print("Model + 91% SFT adapter loaded")
print("VRAM:", round(torch.cuda.memory_allocated()/1e9, 2), "GB")

In [ ]:
# 4. Trajectory tokenizer — assistant-token-only loss masking
# (synced with train_qwen.py's CURRENT tokenize_trajectory_fn: unknown
# roles are masked, truncation drops whole overflowing messages)
from datasets import Dataset

def format_trajectory(messages):
    if not messages:
        return ""
    parts = []
    for m in messages:
        role = m.get("role", "user")
        content = m.get("content", "")
        if role == "system":
            parts.append(f"<|im_start|>system\n{content}<|im_end|>")
        elif role == "user":
            parts.append(f"<|im_start|>user\n{content}<|im_end|>")
        elif role == "assistant":
            parts.append(f"<|im_start|>assistant\n{content}<|im_end|>")
    return "\n".join(parts) + "\n<|im_start|>assistant\n"

def tokenize_trajectory_fn(examples):
    batch_texts, batch_labels = [], []
    for m in examples["messages"]:
        if not m:
            continue
        merged_ids, merged_labels = [], []
        for msg in m:
            role = msg.get("role", "user")
            content = msg.get("content", "")
            if role == "system":
                chunk = f"<|im_start|>system\n{content}<|im_end|>\n"
                train = False
            elif role == "user":
                chunk = f"<|im_start|>user\n{content}<|im_end|>\n"
                train = False
            elif role == "assistant":
                chunk = f"<|im_start|>assistant\n{content}<|im_end|>\n"
                train = True
            else:
                # Unknown roles (e.g. "tool") are context, never trained.
                chunk = f"<|im_start|>{role}\n{content}<|im_end|>\n"
                train = False
            enc = tok(chunk, add_special_tokens=False)
            chunk_ids = enc["input_ids"]
            # Truncate at a message boundary, never mid-message.
            if merged_ids and len(merged_ids) + len(chunk_ids) > TRAJECTORY_CTX:
                break
            merged_ids.extend(chunk_ids)
            merged_labels.extend([c if train else -100 for c in chunk_ids])
        batch_texts.append(merged_ids[:TRAJECTORY_CTX])
        batch_labels.append(merged_labels[:TRAJECTORY_CTX])
    pad = tok.pad_token_id or tok.eos_token_id
    max_len = min(max((len(x) for x in batch_texts), default=TRAJECTORY_CTX), TRAJECTORY_CTX)
    ids_t = torch.full((len(batch_texts), max_len), pad, dtype=torch.long)
    lab_t = torch.full((len(batch_texts), max_len), -100, dtype=torch.long)
    att_t = torch.zeros((len(batch_texts), max_len), dtype=torch.long)
    for i, (ids, labs) in enumerate(zip(batch_texts, batch_labels)):
        ids_t[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)
        lab_t[i, :len(labs)] = torch.tensor(labs, dtype=torch.long)
        att_t[i, :len(ids)] = 1
    return {"input_ids": ids_t, "attention_mask": att_t, "labels": lab_t}

# Task grounding (mirrors train_qwen.py, conditional on trace format):
# old assistant-first traces get an "Issue: ..." user message prepended;
# new traces already start with the real user prompt (role user) and end
# with a trainable assistant SELF-REVIEW — leave them untouched.
grounded = []
for t in traces:
    msgs = t["messages"]
    if msgs and msgs[0].get("role") == "user":
        grounded.append(msgs)
        continue
    grounded.append(
        [{"role": "user", "content": f"Issue: {t['issue']}\n\nStart by exploring the codebase."}]
        + msgs
    )
ds = Dataset.from_dict({"messages": grounded})
tokenized = ds.map(tokenize_trajectory_fn, batched=True,
                    remove_columns=ds.column_names, desc="Tokenizing")
print(f"Tokenized {len(tokenized)} trajectories (grounded with Issue preamble)")
# sanity: assistant tokens should be trainable, user/tool tokens masked
first = tokenized[0]
ntrain = (first['labels'] != -100).sum().item()
print(f"First trace: {first['input_ids'].shape[0]} tokens, {ntrain} trainable (assistant)")


In [ ]:
# 5. Trajectory SFT — Trainer with assistant-only loss
from transformers import (TrainingArguments, Trainer, DataCollatorForSeq2Seq)

out_dir = "qwen_traj_sft"
training_args = TrainingArguments(
    output_dir=out_dir,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=8,  # eff batch 8
    learning_rate=LR,
    warmup_steps=20,
    num_train_epochs=EPOCHS,
    logging_steps=5,
    save_steps=0,
    save_total_limit=1,
    fp16=True,
    remove_unused_columns=False,
    report_to="none",
    dataloader_num_workers=0,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",  # offload optimizer states to CPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=DataCollatorForSeq2Seq(tok, pad_to_multiple_of=8),
)

trainer.train()
trainer.save_model(out_dir)
tok.save_pretrained(out_dir)
print(f"✓ Trajectory SFT done: {out_dir}")

In [ ]:
# 6. Upload the trajectory-SFT adapter to HF
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.upload_folder(
    folder_path=out_dir,
    repo_id=OUT_REPO,
    path_in_repo="traj_sft",
    token=HF_TOKEN,
    ignore_patterns=["*.bin", "optimizer.pt"],
)
print(f"✓ Trajectory SFT adapter uploaded to {OUT_REPO}/traj_sft")
print("\nNext: RLVR with --diversity to bake in creativity.")